## Going Async with Claude Agents

# Unit 24: Asynchronous Multi-Agent Runtimes and Concurrency Topologies

## Introduction & Goals

Welcome to the first lesson of **Parallelizing Claude Agentic Systems in Python**! In the previous modules, you established a production-grade codebase by building an agent orchestrator class capable of managing conversation buffers, executing deterministic tool abstractions, and handling structural routing handoffs to adjacent specialized agents. You have also analyzed the low-level mechanics of the `AsyncAnthropic` client core.

Now, we will take your multi-agent architecture to the next level by re-engineering your core agent orchestrator class to natively drive **multiple concurrent conversations simultaneously** using Python’s async/await concurrency primitives. By the end of this lesson, you will deploy an agent topology that handles numerous distinct Claude API transactions and tool lifecycles concurrently, eliminating idle blocking states and maximizing workflow efficiency.

---

## The Strategic Importance of Async in Multi-Agent Topologies

To understand why asynchronous design is required for advanced multi-agent architectures, we must examine the limitations of synchronous execution. When you dispatch an API call via the standard `Anthropic` client, your execution thread blocks. The program pauses entirely, freezing local resources while waiting for the remote HTTP socket to close.

If your application needs to manage three separate agent conversations, a synchronous setup forces them to execute in a strict, inefficient line:

```text
SYNCHRONOUS (BLOCKING) LIFECYCLE:
┌───────────────────────────┐
│ Turn 1: Fire Async Agent  │ ──► [Thread Blocks / CPU Idle Waiting for API Response] ──► Resolved
└───────────────────────────┘
┌───────────────────────────┐
│ Turn 2: Fire Writer Agent │ ──► [Thread Blocks / CPU Idle Waiting for API Response] ──► Resolved
└───────────────────────────┘
┌───────────────────────────┐
│ Turn 3: Fire Coder Agent  │ ──► [Thread Blocks / CPU Idle Waiting for API Response] ──► Resolved
└───────────────────────────┘

```

This sequential pattern wastes substantial time. While your runtime blocks on a response for Agent 1, it could be initializing connections or executing tools for Agents 2 and 3. Because large language model inference loops are fundamentally I/O-bound operations characterized by high network latency, leaving your local CPU idle during these wait windows creates a major system bottleneck.

Asynchronous programming solves this efficiency gap through cooperative multitasking overseen by a central event loop:

```text
ASYNCHRONOUS (NON-BLOCKING) LIFECYCLE:
 
 ── Dispaching Concurrent Ingestion Matrix ──► [Active Event Loop Engine]
      │                   │                   │
      ▼                   ▼                   ▼
┌───────────┐       ┌───────────┐       ┌───────────┐
│Coroutine 1│       │Coroutine 2│       │Coroutine 3│
└─────┬─────┘       └─────┬─────┘       └─────┬─────┘
      │                   │                   │
      ▼                   ▼                   ▼
[Yields on I/O]     [Yields on I/O]     [Yields on I/O]
      │                   │                   │
      └───────────┬───────┴───────────────────┘
                  ▼
   🔀 Interleaved Socket Returns

```

When an async coroutine hits a network barrier, it explicitly yields control back to the event loop. The loop instantly switches contexts to process other pending tasks—such as firing adjacent requests or parsing incoming data blocks.

For agentic systems, this concurrency means a single coordinator instance can process multiple conversations at the same time, or coordinate a network of specialized agents running in parallel. This becomes incredibly powerful when agents must execute multiple independent tools or manage complex routing handoffs.

---

## Migrating the Orchestrator to the `AsyncAnthropic` Client

Our first refactoring step focuses on the orchestrator's initialization layer. We will replace the standard client gateway with `AsyncAnthropic`. This initialization change shifts the client's return values from immediate, blocking data payloads into awaitable coroutine promises.

Open your `agent.py` core file and update the initialization block:

```python
from anthropic import AsyncAnthropic

class Agent:
    def __init__(
        self,
        name: str,
        system_prompt: str = "You are a helpful assistant.",
        model: str = "claude-sonnet-4-6",
        tools: dict = None,
        tool_schemas: list = None,
        handoffs: list = None,
        max_turns: int = 10
    ):
        # Migrating from Anthropic() to the non-blocking client topology
        self.client = AsyncAnthropic()  
        self.name = name
        self.system_prompt = system_prompt
        self.model = model
        self.tools = tools if tools else {}
        self.tool_schemas = tool_schemas if tool_schemas else []
        self.handoffs = handoffs if handoffs else []
        self.max_turns = max_turns

```

---

## Converting the Core `run` Routine to a Non-Blocking Coroutine

With our async client gateway in place, we must transform our primary conversation tracking routine into an asynchronous coroutine. This requires declaring the method with the `async` keyword and marking our network boundaries with `await`.

Update the `run` method inside `agent.py`:

```python
    async def run(self, input_messages: list) -> tuple[list, str]:
        """
        Drives the multi-turn agent interaction loop asynchronously.
        """
        messages = input_messages.copy()
        turn = 0
        
        while turn < self.max_turns:
            turn += 1
            
            # Use 'await' to yield control to the loop during remote API inference
            response = await self.client.messages.create(
                model=self.model,
                max_tokens=2000,
                system=self.system_prompt,
                tools=self.tool_schemas if self.tool_schemas else None,
                messages=messages
            )
            
            # Capture and preserve structural state transitions
            messages.append({"role": "assistant", "content": response.content})
            
            # Handle tool integration branches
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Intercept explicit multi-agent routing handoffs
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            return handoff_result
                            
                        # (Downstream standard tool execution loops continue here...)
            
            elif response.stop_reason == "end_turn":
                # Return final message history and clean text string
                return messages, response.content[0].text

```

### Deconstructing the Coroutine Execution:

* **`async def`:** Explicitly registers this method block as an asynchronous coroutine inside Python's runtime environment.
* **`await self.client.messages.create(...)`:** Suspends the execution of this specific conversation turn when calling the API, letting the central event loop run other pending tasks while waiting for Claude's response.

---

## Integrating Asynchronous Multi-Agent Routing Handoffs

When an agent decides to route tasks to a downstream expert via a handoff tool call, it must await the completion of that specialist's work. Because the destination agent's `run` function is now an async coroutine, the routing mechanism must also be refactored to support non-blocking wait states.

Update the `call_handoff` pipeline within your orchestrator:

```python
    async def call_handoff(self, tool_use, messages: list) -> tuple[bool, list]:
        """
        Asynchronously hands off conversation history to a specialized downstream agent.
        """
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No explicit rationale provided.")
        
        print(f"🔄 Routing Hand-Off -> Target Specialist: {agent_name}")
        print(f"📝 Transition Rationale     : {reason}")
        
        try:
            # Locate the designated agent instance within the registration list
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            
            # Clean conversation arrays to prevent trailing assistant state blocks
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            
            # Await the execution loop of the target specialist agent safely
            result = await target_agent.run(clean_messages)
            return True, result
            
        except StopIteration:
            error_msg = f"Handoff failed: Target Specialist Agent '{agent_name}' not found."
            print(f"❌ {error_msg}")
            return False, (messages, error_msg)

```

---

## Constructing the Concurrent Ingestion Framework

Now that our agent core is fully async, let's build a runtime orchestration script called `main.py`. This script sets up our tools and uses an async loop to drive multiple independent mathematical conversations in parallel.

```python
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers, multiply_numbers, subtract_numbers, 
    divide_numbers, power, square_root
)

# Load schemas from disk
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Internal execution mapping table
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

```

---

## Deploying Concurrent Tasks via `asyncio.gather`

With our infrastructure in place, we will implement an asynchronous `main` coroutine. This routine prepares our collection of independent prompts, instantiates a single core math agent, and uses `asyncio.gather` to manage multiple active conversations at the same time.

```python
async def main():
    # Define independent, cross-domain prompts to process concurrently
    prompts = [
        "Solve this calculation step by step: (2 + 3) * (4 * 4)",
        "Find the roots of the quadratic equation: x^2 - 5x + 6 = 0",
    ]
    
    # Initialize a single agent configured with our tool matrix
    agent = Agent(
        name="math_assistant",
        system_prompt="You are an expert mathematical agent. Always use tools to verify math operations.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )
    
    print(f"Launching {len(prompts)} Independent Conversations Concurrently...\n")
    
    # Programmatically generate an array of coroutine objects using a list comprehension
    # Note: This instantiates the coroutines in memory but does not start them yet
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]
    
    # Fire all agent loops concurrently and await their complete resolution
    results = await asyncio.gather(*tasks)
    
    # Iterate through the returned matrix to display the final synthesized summaries
    for idx, (history, answer) in enumerate(results, start=1):
        print(f"\n==============================================")
        print(f"📊 CONCURRENT CONVERSATION FLOW OUTPUT: RUN {idx}")
        print(f"==============================================")
        print(answer)

# Bootstrapping the application's central entrypoint loop
if __name__ == "__main__":
    asyncio.run(main())

```

---

## Technical Log Interleaving Analysis

When you execute this parallel pipeline, the stdout logs showcase the non-blocking execution behavior:

```text
Launching 2 Independent Conversations Concurrently...

🔧 Tool called: sum_numbers({'a': 2, 'b': 3})
🔧 Tool called: power({'base': -5, 'exponent': 2})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 4})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 1})
🔧 Tool called: multiply_numbers({'a': 5, 'b': 16})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 6})
...

```

### Key Concurrency Observations:

Notice that the tool calls from completely different conversations are **interleaved (mixed together)** in the console output. Instead of executing the arithmetic expression fully before starting the quadratic equation, the agent switches between tasks dynamically as it waits for network responses. Both tasks run concurrently, allowing the entire pipeline to finish much faster than if they ran sequentially.

---

## Identifying Systemic Bottlenecks: Synchronous Tools

While our agent can now run multiple conversations concurrently, a primary performance bottleneck remains hidden within the system: **the tool execution functions themselves are still synchronous**.

```text
🔧 Tool called: sum_numbers({'a': 2, 'b': 3})   ◄── Thread Blocks Locally
🔧 Tool called: power({'base': -5, 'exponent': 2}) ◄── Waiting for preceding call to release lock

```

Because our internal tool lookup helper (`call_tool()`) runs synchronously, it locks the local thread whenever it executes code. If a tool call requires heavy computing or connects to a slow external API, **all other active conversations freeze** until that specific function releases the thread.

To achieve maximum concurrency across your system, you must also convert your tool execution layers to run asynchronously. We will master this pattern in our next lesson!

---

## Summary & Practice Labs

You have successfully re-engineered your multi-agent architecture to support non-blocking concurrency. By moving to `AsyncAnthropic`, updating your core `run` loops to utilize coroutine structures, and using `asyncio.gather` to manage task execution, you can now run numerous parallel conversations through a single agent instance.

Let's head into the practical code labs to stand up and benchmark your new asynchronous agent architectures!


## Observing Sequential Agent Execution Patterns

Now that you've seen how async can enable concurrency, let's run a version that uses an async entrypoint but still relies on a synchronous agent. This exercise demonstrates a common pitfall: simply wrapping synchronous work in async doesn’t make it concurrent.

You don't need to modify any code for this task—just run the code below and watch the order of the tool calls. Even though main.py uses asyncio and asyncio.gather(), the agent’s run() method is still synchronous and blocks the event loop. As a result, the first conversation completes all of its tool calls before the second one even starts. This is the same sequential, blocking behavior you observed before — just hidden behind an async main.

Compare this to the interleaved tool-call output from the truly async agent in the lesson. The difference shows why real concurrency requires making the inner operations async (e.g., using AsyncAnthropic and awaiting calls inside Agent.run), not just adding an async wrapper at the top level.

```
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


async def run_conversation(agent, prompt):
    # Intentionally call the blocking, synchronous Agent.run() inside an async coroutine.
    # This blocks the event loop until the call completes, so gather() won’t create real concurrency.
    return agent.run([{"role": "user", "content": prompt}])


async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        run_conversation(agent, prompt)
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())

# agent.py
from anthropic import Anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Transfer control to another agent, if the tool use is a handoff
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Execute regular tools
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")
```

## Adding Async Syntax Without Async Client

Now it's time to see what happens when we add async syntax without changing the underlying client. In the lesson, you saw how switching to AsyncAnthropic and making methods async enabled true concurrent execution with interleaved tool calls. But what if we do only half the work?

In this exercise, you'll convert the agent's methods to use async/await syntax while keeping the synchronous Anthropic() client. This will demonstrate an important lesson: just adding async keywords doesn't magically create concurrency if the underlying operations still block.

Your task is to modify the agent.py file by following the TODO comments:

    Make the run() method async
    Make the call_handoff() method async and add await before the target_agent.run() call
    Add await before the call_handoff() call inside the run() method

Important: Do not change Anthropic() to AsyncAnthropic() in the client initialization — this is the whole point of the exercise!

After making these changes, run the code and observe the output. Even though you're using asyncio.gather() and have async methods, the tool calls will execute sequentially rather than being interleaved. This happens because the synchronous Anthropic() client blocks the event loop during API calls, preventing true concurrency. This exercise will help you understand why both async syntax and async-compatible libraries are necessary for concurrent execution.

```
# agent.py
from anthropic import Anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
    
    # TODO: Make this method async by adding the 'async' keyword before 'def'
    def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            # TODO: Add 'await' before this call since target_agent.run() will be async
            result = target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    # TODO: Make this method async by adding the 'async' keyword before 'def'
    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Transfer control to another agent, if the tool use is a handoff
                        if content_item.name == "handoff":
                            # TODO: Add 'await' before this call since call_handoff() will be async
                            handoff_success, handoff_result = self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Execute regular tools
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())
```

Here is the complete, full implementation of both files according to your specifications.

This configuration demonstrates a critical lesson in asynchronous Python development: **adding `async` and `await` keywords alone does not create true concurrency if your underlying I/O operations remain synchronous and blocking.** Because `self.client.messages.create()` inside `agent.py` still invokes the standard synchronous `Anthropic` client library, it locks the operating system thread completely during network operations. As a result, even though `main.py` calls `asyncio.gather(*tasks)`, the operations will still execute one after another in a sequential, blocking fashion instead of interleaving.

### 1. Updated Asynchronous Syntax Agent Core (`agent.py`)

```python
from anthropic import Anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        # INTENTIONALLY KEPT SYNCHRONOUS: This is the core bottleneck for the exercise
        self.client = Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
    
    # TODO: Make this method async by adding the 'async' keyword before 'def'
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            # TODO: Add 'await' before this call since target_agent.run() will be async
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    # TODO: Make this method async by adding the 'async' keyword before 'def'
    async def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            # NOTE: Because self.client is a standard synchronous client, this line 
            # blocks the execution thread completely. There is no 'await' keyword here!
            response = self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Transfer control to another agent, if the tool use is a handoff
                        if content_item.name == "handoff":
                            # TODO: Add 'await' before this call since call_handoff() will be async
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Execute regular tools
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

---

### 2. Core Concurrent Orchestrator Runtime (`main.py`)

```python
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    print("🚀 Launching concurrent scheduling loops with asyncio.gather()...\n")

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())

```

### Architectural Concept Analysis:

When running this layout, you will notice that the `🔧 Tool called:` console trace steps completely execute for `run 1` before `run 2` even wakes up.

Even though you added the `async` / `await` syntax structures across the execution tree, the runtime event loop is never given an opportunity to switch context tasks during the network pass. Because the system calls the synchronous `self.client.messages.create()`, the entire single-threaded Python process freezes, waiting for the HTTP response to hit the socket interface. True asynchronous cooperative multitasking requires **both** async syntactic structure code design **and** an underlying async-compatible library (like `AsyncAnthropic`) to run seamlessly!

## Completing the Async Agent Transformation

In the previous exercise, you discovered that adding async syntax alone isn't enough — the synchronous Anthropic() client still blocks the event loop, preventing true concurrency. Now it's time to complete the transformation and unlock real parallel execution!

Your task is to switch from the synchronous client to the async client in the agent.py file. Follow the TODO comments to make three key changes:

    Change the import statement from Anthropic to AsyncAnthropic.
    Update the client initialization in the __init__ method to use AsyncAnthropic().
    Add the await keyword before the self.client.messages.create() call in the run() method.

Once you've made these changes, run the code and observe how the tool calls from different conversations become interleaved in the output. This interleaving proves that your agent is now truly concurrent — it can switch between conversations while waiting for API responses, making efficient use of time. You have now built a fully async agent system that can handle multiple conversations simultaneously!

```
# TODO: Import the AsyncAnthropic client instead of Anthropic
from anthropic import Anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        # TODO: Switch to AsyncAnthropic client for true concurrent execution
        self.client = Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            # TODO: Add await before this API call to enable concurrent execution
            response = self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Transfer control to another agent, if the tool use is a handoff
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Execute regular tools
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")
```

To fix the blocking bottlenecks and enable true concurrent execution, you need to transition your inner client initialization to `AsyncAnthropic` and ensure all network boundaries are properly awaited. Simply adding the `async` syntax to methods does not create concurrency unless the underlying SDK client drops back into the event loop rather than blocking the thread.

Here is the fully refactored, production-ready `agent.py` script with your modifications complete:

```python
# Import the AsyncAnthropic client instead of Anthropic
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        # Switch to AsyncAnthropic client for true concurrent execution
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # list of other agents to handoff the control
        
        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            # Add await before this API call to enable concurrent execution
            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        # Transfer control to another agent, if the tool use is a handoff
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Execute regular tools
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

```

### Key Architectural Updates Made:

* **Asynchronous Driver Migration:** `self.client` was shifted from `Anthropic()` to `AsyncAnthropic()`, switching all message creation network requests into non-blocking, awaitable coroutines.
* **Non-Blocking Operation Points:** Added the `await` operator directly before the primary execution loop point `self.client.messages.create(...)`. When multiple agents are running via an orchestration wrapper like `asyncio.gather()`, this explicitly signals the event loop to switch context tasks rather than freezing the main process while waiting for the network round-trip.

## Calling Your Async Agent Correctly

## Orchestrating Concurrent Agent Conversations